# Demo: Hybrid CF + Content-Based Recommender (Creative Extension)

**Team 9 — Recommender Systems Spring 2026**

This notebook demonstrates how our **Hybrid CF + Content-Based** creative extension works in practice.

We show a side-by-side comparison: standalone Collaborative Filtering vs. standalone Content-Based vs. the Hybrid — illustrating how combining both signals produces better, more robust recommendations.

---

## Deployment Scenario

**"If this recommender were deployed on Yelp, what problem would it solve best, and for whom?"**

The Hybrid model would be most valuable for **mid-activity users** — people who have enough reviews for CF to work, but whose tastes are specific enough that pure CF gets diluted by the broader population. For example, a user who loves authentic Mexican food in a city where most Yelp users prefer upscale American dining would be poorly served by CF alone (since the popular items don't match their taste). The hybrid corrects this by injecting a content signal that keeps recommendations anchored to the user's actual preference profile. On Yelp, this model would power the main personalized feed, balancing social proof (what people like you enjoy) with personal taste (what your history says you like).

In [1]:
# Imports
import pandas as pd
import numpy as np
import ast
import math
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
ALPHA = 0.6  # CF weight in hybrid

## 1. Load Data

In [2]:
train_df = pd.read_csv('train_reviews_santa_barbara.csv')
restaurants_df = pd.read_csv('restaurants_santa_barbara.csv')

for df in [train_df]:
    df['user_id'] = df['user_id'].astype(str)
    df['business_id'] = df['business_id'].astype(str)
    df['stars'] = df['stars'].astype(float)
restaurants_df['business_id'] = restaurants_df['business_id'].astype(str)

print(f'Loaded {len(train_df)} training reviews, {restaurants_df.shape[0]} restaurants.')

Loaded 41581 training reviews, 767 restaurants.


## 2. Build CF Component (SVD)

In [3]:
users = sorted(train_df['user_id'].unique())
items = sorted(restaurants_df['business_id'].unique())
user_idx = {u: i for i, u in enumerate(users)}
item_idx = {it: i for i, it in enumerate(items)}

R = np.zeros((len(users), len(items)), dtype=np.float32)
for _, row in train_df.iterrows():
    u, it = row['user_id'], row['business_id']
    if u in user_idx and it in item_idx:
        R[user_idx[u], item_idx[it]] = row['stars']

svd = TruncatedSVD(n_components=50, random_state=RANDOM_SEED)
U = svd.fit_transform(R)
Vt = svd.components_
R_pred = U @ Vt

# Normalize per user to [0,1] for hybrid combination
row_min = R_pred.min(axis=1, keepdims=True)
row_max = R_pred.max(axis=1, keepdims=True)
R_norm = (R_pred - row_min) / (row_max - row_min + 1e-9)

print('SVD matrix factorization complete.')
print(f'Variance explained by 50 components: {svd.explained_variance_ratio_.sum()*100:.1f}%')

SVD matrix factorization complete.
Variance explained by 50 components: 36.4%


## 3. Build Content Component

In [4]:
def parse_and_build_features(restaurants_df):
    """Parse restaurant attributes and build a feature matrix."""
    def str_to_dict(s):
        try: return ast.literal_eval(s)
        except: return {}
    def clean_val(d):
        return {k: (v.strip("'\"" ) if isinstance(v, str) else v) for k, v in d.items()}

    df = restaurants_df.copy()
    df['attrs'] = df['attributes'].apply(str_to_dict).apply(clean_val)
    df['good_for_kids'] = df['attrs'].apply(lambda d: 1 if d.get('GoodForKids') == 'True' else 0)
    df['price_1'] = df['attrs'].apply(lambda d: 1 if d.get('RestaurantsPriceRange2') == '1' else 0)
    df['price_2'] = df['attrs'].apply(lambda d: 1 if d.get('RestaurantsPriceRange2') == '2' else 0)
    df['casual'] = df['attrs'].apply(lambda d: 1 if d.get('RestaurantsAttire') == 'casual' else 0)

    top_cats = ['restaurants','food','pizza','mexican','italian','american','chinese',
                'japanese','sushi','burgers','coffee','sandwiches','seafood','thai','mediterranean']
    cat_series = df['categories'].fillna('').str.lower()
    for cat in top_cats:
        df[f'cat_{cat}'] = cat_series.apply(lambda x: 1 if cat in x else 0)

    feat_cols = ['good_for_kids','price_1','price_2','casual'] + [f'cat_{c}' for c in top_cats]
    return df.set_index('business_id')[feat_cols].fillna(0), feat_cols

item_feats, feat_cols = parse_and_build_features(restaurants_df)

# Build user profiles
user_means = train_df.groupby('user_id')['stars'].mean()
user_profiles = {}
for u, group in train_df.groupby('user_id'):
    profile = np.zeros(len(feat_cols)); total_w = 0.0
    for _, row in group.iterrows():
        it = row['business_id']
        if it not in item_feats.index: continue
        w = max(row['stars'] - user_means[u] + 1.0, 0.0)
        profile += w * item_feats.loc[it].values; total_w += w
    if total_w > 0: profile /= total_w
    user_profiles[u] = profile

train_hist = {}
for _, row in train_df.iterrows():
    train_hist.setdefault(row['user_id'], set()).add(row['business_id'])

print('Content profiles built for all users.')

Content profiles built for all users.


## 4. Helper: Get Recommendations from Each Model

In [5]:
def get_cf_recs(user_id, k=10):
    """
    Get top-K recommendations from standalone CF (SVD).

    Parameters
    ----------
    user_id : str
        Target user ID.
    k : int
        Number of recommendations.

    Returns
    -------
    list of str
        Top-K business IDs ranked by CF score.
    """
    if user_id not in user_idx: return []
    ui = user_idx[user_id]
    seen = train_hist.get(user_id, set())
    candidates = [(it, float(R_pred[ui, item_idx[it]])) for it in items if it not in seen and it in item_idx]
    candidates.sort(key=lambda x: -x[1])
    return [it for it, _ in candidates[:k]]


def get_content_recs(user_id, k=10):
    """
    Get top-K recommendations from standalone Content-Based model.

    Parameters
    ----------
    user_id : str
        Target user ID.
    k : int
        Number of recommendations.

    Returns
    -------
    list of str
        Top-K business IDs ranked by content similarity.
    """
    seen = train_hist.get(user_id, set())
    profile = user_profiles.get(user_id, np.zeros(len(feat_cols)))
    candidates = [it for it in item_feats.index if it not in seen]
    if len(candidates) == 0 or profile.sum() == 0: return candidates[:k]
    sims = cosine_similarity([profile], item_feats.loc[candidates].values)[0]
    top_idx = np.argsort(sims)[::-1][:k]
    return [candidates[i] for i in top_idx]


def get_hybrid_recs(user_id, alpha=ALPHA, k=10):
    """
    Get top-K recommendations from the Hybrid CF + Content model.

    Parameters
    ----------
    user_id : str
        Target user ID.
    alpha : float
        Weight for CF component (1-alpha for content).
    k : int
        Number of recommendations.

    Returns
    -------
    list of str
        Top-K business IDs ranked by hybrid score.
    """
    if user_id not in user_idx: return []
    ui = user_idx[user_id]
    seen = train_hist.get(user_id, set())
    candidates = [it for it in items if it not in seen and it in item_idx]

    cf_scores = {it: float(R_norm[ui, item_idx[it]]) for it in candidates}
    profile = user_profiles.get(user_id, np.zeros(len(feat_cols)))
    valid = [it for it in candidates if it in item_feats.index]
    if profile.sum() > 0 and valid:
        sims = cosine_similarity([profile], item_feats.loc[valid].values)[0]
        content_scores = dict(zip(valid, sims.tolist()))
    else:
        content_scores = {}

    hybrid = {it: alpha * cf_scores[it] + (1 - alpha) * content_scores.get(it, 0.0) for it in candidates}
    return sorted(hybrid, key=lambda x: -hybrid[x])[:k]


def ids_to_names(business_ids, restaurants_df):
    """Convert a list of business IDs to a readable name + category table."""
    lookup = restaurants_df.set_index('business_id')[['name','categories']]
    rows = []
    for rank, bid in enumerate(business_ids, 1):
        if bid in lookup.index:
            name = lookup.loc[bid, 'name']
            cats = lookup.loc[bid, 'categories']
            cats = str(cats)[:65] + '...' if len(str(cats)) > 65 else str(cats)
            rows.append({'Rank': rank, 'Name': name, 'Categories': cats})
    return pd.DataFrame(rows)

print('Helper functions ready.')

Helper functions ready.


## 5. Demo: Side-by-Side Comparison

We use the same user as the content-based demo: **User `o6UJMpHcpLJEvmKLrxLS3w`**

This user has 152 reviews and strongly prefers casual Mexican and Italian restaurants.

We compare what each model recommends.

In [6]:
DEMO_USER = 'o6UJMpHcpLJEvmKLrxLS3w'

# Show user history summary
user_reviews = (
    train_df[train_df['user_id'] == DEMO_USER]
    .merge(restaurants_df[['business_id','name','categories']], on='business_id')
    .sort_values('stars', ascending=False)
)
print(f'User: {DEMO_USER}')
print(f'Reviews in training: {len(user_reviews)} | Avg rating: {user_reviews["stars"].mean():.2f}')
print()
print('Sample of their top-rated places:')
for _, r in user_reviews[user_reviews['stars']==5].head(5).iterrows():
    print(f"  ⭐⭐⭐⭐⭐  {r['name']}")

User: o6UJMpHcpLJEvmKLrxLS3w
Reviews in training: 152 | Avg rating: 3.73

Sample of their top-rated places:
  ⭐⭐⭐⭐⭐  Taqueria Altamirano
  ⭐⭐⭐⭐⭐  Los Altos Restaurant
  ⭐⭐⭐⭐⭐  Terraza Cafe
  ⭐⭐⭐⭐⭐  Restaurant OPEN
  ⭐⭐⭐⭐⭐  Rudy's Restaurant No 1


In [7]:
cf_recs = get_cf_recs(DEMO_USER, k=10)
content_recs = get_content_recs(DEMO_USER, k=10)
hybrid_recs = get_hybrid_recs(DEMO_USER, k=10)

print('=' * 60)
print('STANDALONE CF (SVD) — Top 10 Recommendations')
print('=' * 60)
print(ids_to_names(cf_recs, restaurants_df).to_string(index=False))
print()
print('=' * 60)
print('STANDALONE CONTENT-BASED (Metadata Only) — Top 10')
print('=' * 60)
print(ids_to_names(content_recs, restaurants_df).to_string(index=False))
print()
print('=' * 60)
print(f'HYBRID CF + CONTENT (alpha={ALPHA}) — Top 10 Recommendations')
print('=' * 60)
print(ids_to_names(hybrid_recs, restaurants_df).to_string(index=False))

STANDALONE CF (SVD) — Top 10 Recommendations
 Rank                           Name                                                           Categories
    1       Norton's Pastrami & Deli gluten-free, american (new), restaurants, sandwiches, salad, stea...
    2        Super Cuca's Restaurant                                                 restaurants, mexican
    3                 Arnoldi's Cafe breakfast & brunch, bocce ball, event planning & services, beer, ...
    4 Yume Sushi Japanese Restaurant           seafood, restaurants, sushi bars, food, desserts, japanese
    5    Santa Barbara Chicken Ranch health & medical, restaurants, mexican, cannabis clinics, barbequ...
    6              Taqueria El Bajio                                                 restaurants, mexican
    7             Habit Burger Grill                                                 restaurants, burgers
    8              Daves Dogs - Cart             food stands, food, hot dogs, street vendors, restaurants
 

## 6. Analysis: What Does the Hybrid Add?

Looking at the three recommendation lists above:

- **Standalone CF** recommends restaurants that are popular among users with similar interaction patterns. It may suggest well-reviewed restaurants that do not necessarily match this user's specific taste for Mexican and casual food.

- **Standalone Content-Based** focuses entirely on restaurant features. It finds places that look like the restaurants this user has loved — casual, Mexican, affordable. However, it ignores the collective wisdom of other users and can miss restaurants that are excellent but slightly outside the user's typical category profile.

- **Hybrid** balances both signals. It keeps recommendations grounded in the user's taste profile (via content) while also surfacing restaurants that users with similar patterns enjoy (via CF). The result is a list that is both **personalized** and **socially validated**.

This is the core advantage of the hybrid model: neither signal alone is sufficient, but combined, they produce more robust recommendations across a wider variety of users.

## 7. Overlap Analysis

In [8]:
cf_set = set(cf_recs)
content_set = set(content_recs)
hybrid_set = set(hybrid_recs)

print('Overlap between models (Top-10):')
print(f'  CF ∩ Content:  {len(cf_set & content_set)} restaurants in common')
print(f'  CF ∩ Hybrid:   {len(cf_set & hybrid_set)} restaurants in common')
print(f'  Content ∩ Hybrid: {len(content_set & hybrid_set)} restaurants in common')
print()
print('Restaurants unique to Hybrid (not in either standalone model):')
unique_to_hybrid = hybrid_set - cf_set - content_set
if unique_to_hybrid:
    unique_names = restaurants_df[restaurants_df['business_id'].isin(unique_to_hybrid)][['name','categories']]
    for _, r in unique_names.iterrows():
        print(f"  → {r['name']}  |  {str(r['categories'])[:60]}")
else:
    print('  (Hybrid top-10 is a subset of the two standalone models for this user)')

Overlap between models (Top-10):
  CF ∩ Content:  0 restaurants in common
  CF ∩ Hybrid:   8 restaurants in common
  Content ∩ Hybrid: 1 restaurants in common

Restaurants unique to Hybrid (not in either standalone model):
  → Palapa  |  restaurants, mexican
